In [1]:
import subprocess
subprocess.run(['pip', 'install', '--upgrade', 'pydantic', 'groq', 'PyPDF2'], check=True)
print('✅ Done — now go to Kernel → Restart Kernel, then run your cells again')

✅ Done — now go to Kernel → Restart Kernel, then run your cells again


In [2]:
!pip install groq PyPDF2 --quiet
print('✅ Done')

✅ Done


In [3]:
!pip install google-genai PyPDF2 --quiet
print('✅ Done')

✅ Done


In [6]:
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


In [7]:
try:
    from google.colab import files
    print('📂 Upload INSPECTION REPORT PDF:')
    uploaded_insp = files.upload()
    print('📂 Upload THERMAL IMAGES PDF:')
    uploaded_therm = files.upload()
    insp_bytes  = list(uploaded_insp.values())[0]
    therm_bytes = list(uploaded_therm.values())[0]
    insp_name   = list(uploaded_insp.keys())[0]
    therm_name  = list(uploaded_therm.keys())[0]
except ImportError:
    INSPECTION_PDF_PATH = "Sample Report.pdf"   # change if needed
    THERMAL_PDF_PATH    = "Thermal Images.pdf"  # change if needed
    with open(INSPECTION_PDF_PATH, 'rb') as f: insp_bytes = f.read()
    with open(THERMAL_PDF_PATH, 'rb') as f:    therm_bytes = f.read()
    insp_name, therm_name = INSPECTION_PDF_PATH, THERMAL_PDF_PATH

print(f'✅ Inspection : {insp_name}  ({len(insp_bytes)//1024} KB)')
print(f'✅ Thermal    : {therm_name}  ({len(therm_bytes)//1024} KB)')

✅ Inspection : Sample Report.pdf  (1119 KB)
✅ Thermal    : Thermal Images.pdf  (3890 KB)


In [8]:
# ── ALL imports ──
import io, re
import PyPDF2
from groq import Groq
from datetime import datetime
from IPython.display import display, HTML, FileLink

# ── Get free API key at console.groq.com (no credit card needed) ──
groq_client = Groq(api_key="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")

# ── Functions ──
def extract_text(pdf_bytes, label='Doc'):
    reader = PyPDF2.PdfReader(io.BytesIO(pdf_bytes))
    out = [f'[{label} — {len(reader.pages)} pages]']
    for i, page in enumerate(reader.pages, 1):
        t = page.extract_text() or ''
        if t.strip():
            out.append(f'\n--- Page {i} ---\n{t}')
    return '\n'.join(out)

def detect_issues(insp, therm):
    conflicts, missing = [], []
    i = insp.lower()
    if 'nahani trap' in i and 'yes' in i and 'no' in i:
        conflicts.append('Nahani trap status shows both Yes and No — needs clarification')
    if 'all time' in i and 'monsoon' in i:
        conflicts.append('Leakage timing shows both All time and Monsoon — likely different areas; verify')
    insp_dates  = re.findall(r'\d{2}[./]\d{2}[./]\d{4}', insp)
    therm_dates = re.findall(r'\d{2}[./]\d{2}[./]\d{4}', therm)
    if insp_dates and therm_dates and insp_dates[0] != therm_dates[0]:
        conflicts.append(f'Date mismatch: Inspection {insp_dates[0]} vs Thermal {therm_dates[0]}')
    for kw in ['customer name', 'client name', 'owner']:
        if kw not in i:
            missing.append('Customer / Owner Name')
            break
    if 'not sure' in i:
        missing.append('External Paint Type (inspector marked Not Sure)')
    if 'terrace' not in i:
        missing.append('Terrace / Roof Inspection Data')
    return {'conflicts': list(set(conflicts)), 'missing': list(set(missing))}

def generate_ddr(insp_text, therm_text, issues):
    conflict_block = ('\n⚠️ CONFLICTS:\n' + '\n'.join(f'  • {c}' for c in issues['conflicts'])) if issues['conflicts'] else ''
    missing_block  = ('\n⚠️ POSSIBLY MISSING:\n' + '\n'.join(f'  • {m}' for m in issues['missing'])) if issues['missing'] else ''
    today = datetime.now().strftime('%d %B %Y')

    prompt = f"""You are a senior building inspection engineer at UrbanRoof Private Limited.
Generate accurate, structured, client-friendly Detailed Diagnostic Reports.
Never invent facts. Mark conflicts explicitly. Write Not Available for missing data.

=== INSPECTION REPORT ===
{insp_text[:4500]}

=== THERMAL IMAGING REPORT ===
{therm_text[:3500]}

{conflict_block}
{missing_block}

Generate a complete DDR using EXACTLY this structure:

# DETAILED DIAGNOSTIC REPORT (DDR)
**Report Date:** {today}
**Inspection Date:** [from documents]
**Inspected By:** [from documents]
**Property Type:** [from documents]
**Inspection Score:** [from documents]

## 1. PROPERTY ISSUE SUMMARY
[Executive summary — total areas, causes, urgency]

## 2. AREA-WISE OBSERVATIONS
[For each area: Symptom / Root Source / Thermal Finding / Temperature Differential / Image Ref]

## 3. PROBABLE ROOT CAUSE
[Table: Area | Root Cause | Evidence Source]

## 4. SEVERITY ASSESSMENT
[Table: Area | Severity (HIGH/MODERATE/LOW) | Priority | Reasoning]

## 5. RECOMMENDED ACTIONS
### Priority 1 — Immediate (0-4 Weeks)
### Priority 2 — Secondary (1-3 Months)
### Priority 3 — Preventive (3-6 Months)

## 6. ADDITIONAL NOTES
[Important warnings and observations]

## 7. MISSING OR UNCLEAR INFORMATION
[Table: Field | Status | Note]

RULES: No invented facts. Label CONFLICT where found. Write Not Available for missing info."""

    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",   # free, very capable
        messages=[
            {"role": "system", "content": "You are a senior building inspection engineer. Generate accurate structured DDR reports."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=4096
    )
    return resp.choices[0].message.content

# ── Run pipeline ──
print('📖 Extracting text from PDFs...')
insp_text  = extract_text(insp_bytes,  'Inspection Report')
therm_text = extract_text(therm_bytes, 'Thermal Report')

print('🔍 Detecting conflicts...')
issues = detect_issues(insp_text, therm_text)
if issues['conflicts']: print(f'  ⚠️  {len(issues["conflicts"])} conflict(s) detected')
if issues['missing']:   print(f'  ℹ️  {len(issues["missing"])} missing field(s)')

print('🤖 Generating DDR with Groq (free)...')
ddr_text = generate_ddr(insp_text, therm_text, issues)

print('\n✅ DDR GENERATED\n' + '='*70)
print(ddr_text)
print('='*70)

# ── Save HTML ──
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>DDR — UrbanRoof</title>
<style>
  body{{ font-family:Arial,sans-serif; max-width:900px; margin:0 auto; padding:30px; color:#222; }}
  h1{{ color:#1a1a2e; border-bottom:4px solid #e6007a; padding-bottom:8px; }}
  h2{{ color:#16213e; border-left:5px solid #e6007a; padding-left:12px; }}
  h3{{ color:#0f3460; }}
  table{{ border-collapse:collapse; width:100%; margin:12px 0; }}
  th{{ background:#1a1a2e; color:white; padding:10px; text-align:left; }}
  td{{ border:1px solid #ddd; padding:8px 12px; }}
  tr:nth-child(even) td{{ background:#f7f7f7; }}
  .header{{ background:linear-gradient(135deg,#1a1a2e,#0f3460);
             color:white; padding:24px; border-radius:10px; margin-bottom:28px; }}
</style></head>
<body>
<div class="header">
  <h1 style="color:white;border:none;">UrbanRoof — Detailed Diagnostic Report</h1>
  <p>Generated: {datetime.now().strftime('%d %B %Y %H:%M')}</p>
</div>
<pre style="white-space:pre-wrap;line-height:1.7;font-family:inherit;">{ddr_text}</pre>
<hr>
<p style="text-align:center;color:#888;font-size:.8rem;">
  AI-Generated by UrbanRoof DDR System · Powered by Groq + Llama 3.3
</p>
</body></html>"""

output_file = f'DDR_Report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.html'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html_content)
print(f'✅ Saved: {output_file}')

try:
    from google.colab import files
    files.download(output_file)
    print('📥 Downloading...')
except ImportError:
    display(FileLink(output_file))
    print('💾 Click the link above to download')

display(HTML(f'<b>Preview:</b><br><pre>{html_content[:3000]}</pre>'))

📖 Extracting text from PDFs...


Multiple definitions in dictionary at byte 0x333977 for key /Im28
Multiple definitions in dictionary at byte 0x333991 for key /Im30
Multiple definitions in dictionary at byte 0x3339ab for key /Im32
Multiple definitions in dictionary at byte 0x3339c5 for key /Im34
Multiple definitions in dictionary at byte 0x3339f9 for key /Im38
Multiple definitions in dictionary at byte 0x333a13 for key /Im40
Multiple definitions in dictionary at byte 0x333a2d for key /Im42
Multiple definitions in dictionary at byte 0x333a47 for key /Im44
Multiple definitions in dictionary at byte 0x333a7b for key /Im48
Multiple definitions in dictionary at byte 0x333a95 for key /Im50
Multiple definitions in dictionary at byte 0x333aaf for key /Im52
Multiple definitions in dictionary at byte 0x333ac9 for key /Im54
Multiple definitions in dictionary at byte 0x333afd for key /Im58
Multiple definitions in dictionary at byte 0x333b17 for key /Im60
Multiple definitions in dictionary at byte 0x333b31 for key /Im62
Multiple d

🔍 Detecting conflicts...
  ℹ️  3 missing field(s)
🤖 Generating DDR with Groq (free)...

✅ DDR GENERATED
 
Introduction  
 
Introduction to thermal imaging and its application 
 
--- Page 2 ---
 
Background  Thermal imaging,  also known as thermography,  
 
is a technique used to detect temperature differences  
 
in objects or surfaces 
Thermal imaging can be used for predictive maintenance,  
energy auditing, 
building diagnostics, and condition assessment 
Objectives  
 
The objectives of this thermal imaging survey 
were to identify any abnormalities 
in the building envelope, such as air leaks, 
moisture intrusion, and heat loss 
Equipment  Used  Testo  875 thermal imaging camera 
 
--- Page 3 ---
 
 Methodology  
 
The thermal imaging survey was conducted on September 27, 
2022 
The building was surveyed both internally and externally 
using the Testo 875 thermal imaging camera 
 
--- Page 4 ---
 
Results  
 
The thermal imaging survey revealed several areas of 
concern, including

C:\Users\Lenovo\Downloads\files\DDR_Report_20260626_164932.html

💾 Click the link above to download
